# Two-Qubit Post-Processing: Fidelity Evaluation

This notebook post-processes the reconstructed Choi matrices from the two-qubit
process tomography experiment collected by `SimulationTwoQubit.ipynb` to produce
the process fidelities reported in the manuscript.

For each of the 40 metric operators the notebook:
1. Loads the reconstructed Choi matrix from `data/raw_tomography/qubit_choi_matrices/`.
2. Projects the Choi matrix onto the set of physical (positive semidefinite) operators.
3. Computes the process fidelity against the ideal Choi matrix and compares against
   the adversarial threshold.

Results are saved to `data/processed/experimental_fidelities_qubit.csv`.

In [ ]:
import sys
import os
from math import pi

import numpy as np
import pandas as pd
import scipy as sp

from forest.benchmarking.tomography import proj_choi_to_physical

if sys.version_info < (3, 7):
    from pyquil.external.dataclasses import dataclass
else:
    from dataclasses import dataclass

## Metric operator parameter grid

In [ ]:
def eta_grid():
    """Generate a grid of 40 metric operator parameter vectors [r, theta, phi, alpha].

    The first 20 points sample two delta values at five nu values with phi = -pi/2.
    The next 20 points are related to the first 20 by a Hadamard transform, with
    phi = pi/2 and theta = pi/2.

    Returns:
        list of [r, theta, phi, alpha] parameter vectors (length 40).
    """
    grid = []
    nu_array = np.linspace(0.05, 0.95, 5)

    # First 20 points: phi = -pi/2 and pi/2, theta varies with delta
    for delta_array, phi in [
        (np.linspace(-pi/4, pi/4, 2), -pi/2),
        (np.linspace(3*pi/4, 5*pi/4, 2), pi/2),
    ]:
        for nu in nu_array:
            for delta in delta_array:
                r = (1 - nu) / (1 + nu)
                theta = pi/2 - delta if phi == -pi/2 else -pi/2 + delta
                alpha = 2 * np.arccos(np.sqrt(1 - np.abs(r)))
                grid.append([r, theta, phi, alpha])

    # Last 20 points: theta = pi/2, phi varies with delta (Hadamard-related metrics)
    delta_array = np.linspace(pi/4, 7*pi/4, 4)
    for nu in nu_array:
        for delta in delta_array:
            r = (1 - nu) / (1 + nu)
            theta = pi / 2
            phi = 3*pi/2 - delta
            alpha = 2 * np.arccos(np.sqrt(1 - np.abs(r)))
            grid.append([r, theta, phi, alpha])

    return grid

## Helper functions

In [ ]:
def get_etahalf(r, theta, phi):
    """Construct the square root of the metric operator eta from its parameters.

    Args:
        r: radial metric parameter.
        theta: polar angle parameter (radians).
        phi: azimuthal angle parameter (radians).

    Returns:
        eta_half: (2, 2) complex array, the matrix square root of the metric operator.
    """
    eta_half = np.array([
        [
            np.cos(theta/2)**2 + np.sqrt(r) * np.sin(theta/2)**2,
            (np.cos(phi) - 1j*np.sin(phi)) * np.cos(theta/2) * np.sin(theta/2) * (1 - np.sqrt(r))
        ],
        [
            (np.cos(phi) + 1j*np.sin(phi)) * np.cos(theta/2) * np.sin(theta/2) * (1 - np.sqrt(r)),
            np.sqrt(r) * np.cos(theta/2)**2 + np.sin(theta/2)**2
        ]
    ])
    return eta_half


def get_idealchoi(eta_half):
    """Compute the ideal Choi matrix for the IP-change channel defined by eta_half.

    Args:
        eta_half: (2, 2) complex array, the matrix square root of the metric operator.

    Returns:
        choi: (4, 4) complex array, the ideal Choi matrix.
    """
    max_entangled = np.array([[1, 0, 0, 1]]).T / np.sqrt(2)
    identity = np.eye(2)
    A = np.kron(identity, eta_half)
    choi = 2 * A @ max_entangled @ max_entangled.conj().T @ A.conj().T
    return choi


def get_fidelity(ideal_choi, estimated_choi):
    """Compute the process fidelity between the ideal and estimated Choi matrices.

    Uses the quantum fidelity formula F = Tr[sqrt(sqrt(A) B sqrt(A))] / (sqrt(Tr[A]) sqrt(Tr[B])).

    Args:
        ideal_choi: (4, 4) complex array, the ideal Choi matrix.
        estimated_choi: (4, 4) complex array, the experimentally reconstructed Choi matrix.

    Returns:
        fidelity: real scalar process fidelity.
    """
    sqrt_ideal = sp.linalg.sqrtm(ideal_choi)
    numerator = np.trace(sp.linalg.sqrtm(sqrt_ideal @ estimated_choi @ sqrt_ideal))
    denominator = np.sqrt(np.trace(ideal_choi)) * np.sqrt(np.trace(estimated_choi))
    return (numerator / denominator).real

## Evaluate fidelities for all 40 metric operators

The Choi matrices were already reconstructed via linear inversion in
`SimulationTwoQubit.ipynb`. Here we project each to the nearest physical
(positive semidefinite) operator before computing the process fidelity.

In [ ]:
INPUT_DIR = "data/raw_tomography/qubit_choi_matrices"
OUTPUT_DIR = "data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

grid = eta_grid()
rows = []

for k in range(40):
    # Load reconstructed Choi matrix
    choi_path = os.path.join(INPUT_DIR, f"qubit_choi_metric_{k+1:02d}.npy")
    choi = np.load(choi_path)

    # Project to nearest physical (positive semidefinite) operator
    choi_proj = proj_choi_to_physical(choi, False)

    # Compute process fidelity against ideal Choi matrix
    eta_half = get_etahalf(grid[k][0], grid[k][1], grid[k][2])
    ideal_choi = get_idealchoi(eta_half)
    fidelity = get_fidelity(ideal_choi, choi_proj)

    # Adversarial threshold for certifying a genuinely higher-dimensional process
    r = grid[k][0]
    threshold = (1 + np.sqrt(r)) / (np.sqrt(2) * np.sqrt(1 + r))

    rows.append({
        'index': k + 1,
        'r': r,
        'theta_pi': grid[k][1] / pi,
        'phi_pi': grid[k][2] / pi,
        'threshold_fidelity': round(threshold, 3),
        'qubit_fidelity': round(fidelity, 3),
        'qubit_passed': fidelity > threshold,
    })

# Save results
out_path = os.path.join(OUTPUT_DIR, "experimental_fidelities_qubit.csv")
pd.DataFrame(rows).to_csv(out_path, index=False)
print(f"Done. Results saved to {out_path}")
print(f"Metrics passing threshold: {sum(r['qubit_passed'] for r in rows)}/40")